In [ ]:
from google.colab import files
import pandas as pd, numpy as np, matplotlib.pyplot as plt
f = next(iter(files.upload()))   # choose QC_FIX.xlsx (or your updated workbook)

In [ ]:
CODE, VS, L = 'IDR', 'SGD', 2          # selected currency, comparison currency, sigma multiplier

w = pd.read_excel(f, 'Data Dunia', index_col=0)
w = w.iloc[:, :next(i for i, c in enumerate(w.columns) if str(c).startswith('Unnamed'))]   # raw rates only, stop at first blank header
w.index = pd.to_datetime(w.index, errors='coerce')
w = w[w.index.notna()]                              # drop summary rows below the data
z = (w - w.mean()) / w.std()                       # z-score per currency, sample sd
lim = L * np.sqrt((len(w) - 1) / len(w))           # control limits on the z scale

def chart(ax, s, title, u=None, l=None, cl=None):
    cl = s.mean() if cl is None else cl
    u = cl + L * s.std() if u is None else u
    l = cl - L * s.std() if l is None else l
    out = (s > u) | (s < l)
    ax.plot(s.index, s, color='#2563a8', lw=1.6)
    ax.scatter(s.index[out], s[out], color='crimson', zorder=3)
    ax.axhline(cl, color='gray'); ax.axhline(u, color='crimson', ls='--'); ax.axhline(l, color='crimson', ls='--')
    ax.set_title(f'{title}  ({out.sum()} of {len(s)} points out of control)')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
chart(ax, w[CODE], f'{CODE} per SDR')
plt.show()

In [ ]:
last = z.iloc[-1].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(last)), last.values, color='#2563a8', lw=.8, marker='o', ms=3)
for i, (c, v) in enumerate(last.items()):
    if abs(v) > lim: ax.plot(i, v, 'o', color='crimson')
    if c in (CODE, VS): ax.annotate(c, (i, v), xytext=(0, 8), textcoords='offset points', ha='center', weight='bold')
ax.axhline(0, color='gray'); ax.axhline(lim, color='crimson', ls='--'); ax.axhline(-lim, color='crimson', ls='--')
ax.set_title(f'World control chart, {z.index[-1]:%b %Y}'); ax.set_xticks([])
plt.show()
print(f'{CODE}: z = {last[CODE]:+.2f}, rank {last.index.get_loc(CODE) + 1} of {len(last)} codes (incl. metals and crypto)')

In [ ]:
yr = z.groupby(z.index.year).agg(['mean', 'count'])
full = [y for y in yr.index if yr.loc[y].xs('count', level=1).iloc[0] == 12]
fig, ax = plt.subplots(figsize=(8, 4))
for c in (CODE, VS): ax.plot([str(y) for y in full], [yr.loc[y, (c, 'mean')] for y in full], marker='o', label=c)
ax.axhline(0, color='gray'); ax.axhline(lim, color='crimson', ls='--'); ax.axhline(-lim, color='crimson', ls='--')
ax.set_title('Yearly average z-score'); ax.legend()
plt.show()